# TRICE — Business Entity Resolution (Kaggle runner)

**Team Vortex** · Aryan Bansal (lead) · Sachin Kumar · Daksh Tandon · Parth Aggarwal

Runs the full TRICE pipeline on Kaggle: prepare -> mine -> train -> tune -> infer, then
writes `matching_results.tsv` and `candidate_pairs.tsv` to `/kaggle/working/output/`.

## Before you run

1. **Add the code**: this notebook clones `https://github.com/iittjjee2024/TRICE.git`.
   No token needed for a public repo.
2. **Add the dataset**: *Add Input* -> *Datasets* -> your uploaded challenge dataset.
   Upload the seven TSVs preserving the `train/` and `test/` folders. The notebook
   auto-detects the mount under `/kaggle/input/`; if detection fails, set `DATA_DIR`
   manually in the config cell.
3. **Settings**: Accelerator *None* (CPU is fine; the models are gradient-boosted trees),
   Internet *On* (for the clone + pip). Persistence is optional.
4. Run all. On the full test set inference takes roughly 1.5–2.5 h; use the `SUBSET`
   knob in the config cell for a fast smoke run first.

In [ ]:
# --- clone the code -------------------------------------------------------------
import os, subprocess, sys

REPO = "https://github.com/iittjjee2024/TRICE.git"
CODE = "/kaggle/working/TRICE"
if not os.path.isdir(CODE):
    subprocess.run(["git", "clone", "--depth", "1", REPO, CODE], check=True)
else:
    subprocess.run(["git", "-C", CODE, "pull", "--ff-only"], check=False)
print("code at", CODE)
print(sorted(os.listdir(CODE)))

In [ ]:
# --- dependencies ---------------------------------------------------------------
# Kaggle already ships numpy/pandas/scipy/scikit-learn/pyarrow/lightgbm. We only need
# the two light packages the pipeline adds. Versions are left to Kaggle's stack to avoid
# a slow, conflict-prone full reinstall.
!pip install -q rapidfuzz Unidecode
import numpy, pandas, sklearn, scipy, rapidfuzz, lightgbm, pyarrow, unidecode
print("numpy", numpy.__version__, "| pandas", pandas.__version__,
      "| sklearn", sklearn.__version__, "| lightgbm", lightgbm.__version__)

In [ ]:
# --- configuration --------------------------------------------------------------
import glob, os

CODE = "/kaggle/working/TRICE"
WORK = "/kaggle/working/trice_run"       # writable: store, model, artifacts
OUT  = "/kaggle/working/output"          # the two submission TSVs land here
os.makedirs(WORK, exist_ok=True)
os.makedirs(OUT, exist_ok=True)

# Fast smoke run vs full run. Set SUBSET=None for the real submission.
SUBSET = 4000        # entities/country for train; None = full
TRAIN_ENTITIES = 70000 if SUBSET is None else SUBSET

# If auto-detection fails, set this to the folder that directly contains the seven TSVs
# (or their train/ and test/ subfolders), e.g. "/kaggle/input/amazon-ml-2026".
DATA_DIR_OVERRIDE = None

# The seven files we need. Kaggle often FLATTENS the train/ and test/ folders on upload,
# so we locate each file by name anywhere under /kaggle/input rather than assuming a
# fixed folder layout.
REQUIRED = [
    ("train", "train_source1.tsv"), ("train", "train_source2.tsv"),
    ("train", "train_source3.tsv"), ("train", "train_ground_truth.tsv"),
    ("test", "test_source1.tsv"), ("test", "test_source2.tsv"),
    ("test", "test_source3.tsv"),
]

def locate_files():
    """Return {(split, filename): absolute path} by globbing for each name."""
    found = {}
    for split, fname in REQUIRED:
        hits = glob.glob(f"/kaggle/input/**/{fname}", recursive=True)
        if hits:
            found[(split, fname)] = sorted(hits, key=len)[0]  # shallowest match
    return found

found = locate_files()
missing = [f"{s}/{f}" for (s, f) in REQUIRED if (s, f) not in found]

if missing:
    print("MISSING:", missing)
    print("\nEverything currently under /kaggle/input:")
    any_tsv = False
    for p in sorted(glob.glob("/kaggle/input/**/*", recursive=True)):
        if os.path.isfile(p):
            print("  ", p)
            any_tsv = any_tsv or p.endswith(".tsv")
    if not any_tsv:
        print("  (no files at all — the dataset is not attached; use 'Add Input')")
    raise SystemExit(
        "Dataset files not found. Attach the challenge dataset via 'Add Input', or set "
        "DATA_DIR_OVERRIDE above to the folder shown in the listing.")

# Build a clean train/ + test/ layout in the writable working dir by symlinking the
# located files, so the pipeline sees exactly the structure it expects regardless of how
# the dataset was uploaded.
DATA_DIR = os.path.join(WORK, "dataset")
for split in ("train", "test"):
    os.makedirs(os.path.join(DATA_DIR, split), exist_ok=True)
for (split, fname), src in found.items():
    dst = os.path.join(DATA_DIR, split, fname)
    if os.path.islink(dst) or os.path.exists(dst):
        os.remove(dst)
    os.symlink(os.path.realpath(src), dst)

if DATA_DIR_OVERRIDE:
    DATA_DIR = DATA_DIR_OVERRIDE
print("DATA_DIR =", DATA_DIR)
for split in ("train", "test"):
    for fname in sorted(os.listdir(os.path.join(DATA_DIR, split))):
        p = os.path.join(DATA_DIR, split, fname)
        print(f"  {split}/{fname:<24} {os.path.getsize(p)/1e6:8.1f} MB")

In [ ]:
# --- wire the repo paths to the Kaggle layout -----------------------------------
# The scripts resolve paths relative to the repo root: student_resource/dataset for the
# data and artifacts/ for outputs. On Kaggle the repo root is read-only-ish and the input
# is elsewhere, so we symlink the expected locations to the real ones.
import os

def link(src, dst):
    os.makedirs(os.path.dirname(dst), exist_ok=True)
    if os.path.islink(dst) or os.path.exists(dst):
        if os.path.islink(dst):
            os.unlink(dst)
        else:
            return
    os.symlink(src, dst)

link(DATA_DIR, os.path.join(CODE, "student_resource", "dataset"))
# put the heavy generated artifacts on the writable working disk
art = os.path.join(WORK, "artifacts")
os.makedirs(art, exist_ok=True)
link(art, os.path.join(CODE, "artifacts"))
link(OUT, os.path.join(CODE, "output"))
print("linked dataset ->", os.path.realpath(os.path.join(CODE, 'student_resource', 'dataset')))
print("linked artifacts ->", os.path.realpath(os.path.join(CODE, 'artifacts')))

In [ ]:
# --- correctness tests (fast, need no dataset) ----------------------------------
import subprocess
PY = sys.executable
for t in ("test_decide.py", "test_union.py"):
    print("=" * 70, "\n", t)
    subprocess.run([PY, os.path.join(CODE, "scripts", t)], cwd=CODE, check=True)

In [ ]:
# --- helper to stream a stage's output ------------------------------------------
import subprocess, sys, time
PY = sys.executable

def run(stage_args, title):
    print("=" * 78, f"\n{title}\n", "=" * 78, flush=True)
    t0 = time.time()
    p = subprocess.Popen([PY, *stage_args], cwd=CODE, stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, bufsize=1,
                         env={**os.environ, "PYTHONUNBUFFERED": "1"})
    for line in p.stdout:
        print(line, end="")
    p.wait()
    print(f"\n[{title}] exit={p.returncode}  {time.time()-t0:.0f}s", flush=True)
    if p.returncode != 0:
        raise RuntimeError(f"{title} failed")

In [ ]:
# --- stage 1: mine token aliases from the training ground truth -----------------
run(["scripts/02_mine_variants.py", "--sample", "250000"], "mine variants")

In [ ]:
# --- stage 2: normalise all records into the Parquet record store ---------------
# Kaggle CPU has 4 cores; use 3 workers.
run(["scripts/03_prepare.py", "--workers", "3"], "prepare record store")

In [ ]:
# --- stage 3: train the matcher + score a held-out validation split -------------
run(["scripts/05_train.py", "--entities", str(TRAIN_ENTITIES), "--run-id", "kaggle"],
    "train + validate")

In [ ]:
# --- stage 4: search the decision-layer configuration on validation -------------
run(["scripts/07_tune_decision.py", "--run-id", "kaggle"], "tune decision")

In [ ]:
# --- inspect the validation metrics ---------------------------------------------
import json
m = json.load(open(os.path.join(CODE, "artifacts", "runs", "kaggle", "metrics.json")))
print("model:", m["model_kind"], "| use_stage2:", m["use_stage2"])
print("stage1:", m["stage1"])
print("stage2:", m["stage2"])
ef = m["decision_rules"]["expected_f"]
print(f"VAL macro F0.5 = {ef['val_macro_f05']:.5f}  "
      f"P={ef['val_macro_precision']:.4f}  R={ef['val_macro_recall']:.4f}")
print("per-country:", m.get("decision_tuning", {}).get("by_country"))

In [ ]:
# --- stage 5: full test-set inference -> output/*.tsv ---------------------------
# Kaggle gives ~30 GB RAM (more than the 16 GB dev box), so the default query batch is
# safe; lower --query-batch if you hit memory limits.
run(["scripts/06_infer.py", "--run-id", "kaggle", "--query-batch", "120000"],
    "full test inference")

In [ ]:
# --- validate + preview the submittable TSV -------------------------------------
import subprocess
sub = subprocess.run(
    [PY, "student_resource/utils/validate_submission.py",
     "--matching", "output/matching_results.tsv",
     "--candidate", "output/candidate_pairs.tsv",
     "--test-dir", "student_resource/dataset/test"],
    cwd=CODE, capture_output=True, text=True)
print(sub.stdout)
print(sub.stderr)

print("=" * 60, "\nhead of matching_results.tsv:")
with open(os.path.join(OUT, "matching_results.tsv"), encoding="utf-8") as f:
    for i, line in zip(range(6), f):
        print(line.rstrip())

In [ ]:
# --- surface the outputs in /kaggle/working for download ------------------------
# Files under /kaggle/working are attached to the notebook's output; download
# matching_results.tsv from the Output tab and upload it to the challenge portal.
import shutil
for f in ("matching_results.tsv", "candidate_pairs.tsv"):
    src = os.path.join(OUT, f)
    if os.path.isfile(src):
        shutil.copy(src, os.path.join("/kaggle/working", f))
        print(f"{f}: {os.path.getsize(src)/1e6:.1f} MB -> /kaggle/working/{f}")